In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

df = pd.read_csv("../outputs/appointments_clean.csv")

cols = ["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows",
        "reminder_sent", "booking_channel", "visit_type"]
X = df[cols].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]

print(X.shape[1], list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(len(X_train), len(X_test))
print(y_test.sum())
print(round((y_test == 0).mean(), 4), (y_test == 0).sum())

11 ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
792 198
49
0.7525 149


In [2]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

pred = model.predict(X_test_s)
prob = model.predict_proba(X_test_s)[:, 1]

print("accuracy", round(accuracy_score(y_test, pred), 3))
print(confusion_matrix(y_test, pred))
print("precision", round(precision_score(y_test, pred), 2))
print("recall", round(recall_score(y_test, pred), 2))
print("f1", round(f1_score(y_test, pred), 2))
print("roc_auc", round(roc_auc_score(y_test, prob), 3))

accuracy 0.788
[[139  10]
 [ 32  17]]
precision 0.63
recall 0.35
f1 0.45
roc_auc 0.789


In [3]:
for name, c in zip(X.columns, model.coef_[0]):
    print(name, round(c, 2))

i = list(X.columns).index("reminder_sent")
coef = model.coef_[0][i]
sd = scaler.scale_[i]

print("coef", round(coef, 2), "sd", round(sd, 2))
print("exp(coef)", round(np.exp(coef), 2))
print("exp(coef / sd)", round(np.exp(coef / sd), 2))

age -0.25
distance_km 0.25
fee_inr 0.45
lead_days 0.61
prior_no_shows 0.65
reminder_sent -0.58
booking_channel_App -0.05
booking_channel_Phone 0.22
booking_channel_Walk-In -0.19
visit_type_Consultation 0.02
visit_type_Follow-up -0.02
coef -0.58 sd 0.49
exp(coef) 0.56
exp(coef / sd) 0.31


### Task 3.2 – Reminder effect

The coefficient for reminder_sent is -0.58, the 
most negative of all features. np.exp(-0.58) = 
0.56, but this is the odds change for one standard 
deviation of the scaled feature (SD ≈ 0.49), so 
this is not a real world change.


To get the effect of a reminder going from No (0) 
to Yes (1), I divided by the standard deviation 
first: np.exp(-0.58 / 0.49) = 0.31. In this model, 
holding the other features constant, appointments 
with a reminder have about 69% lower odds of a no-
show.


The two numbers differ because the model was fitted 
on scaled features, and a full 0 to 1 change is 
about two standard deviations, so the real effect 
is stronger than the per-SD figure.

This is an association, not proof of cause: the 
data are observational, so reminders may have been 
sent to patients who were already more likely to 
attend.

In [4]:
from sklearn.tree import DecisionTreeClassifier

full = DecisionTreeClassifier(random_state=42)
full.fit(X_train, y_train)
print("full depth", full.get_depth())
print("train", round(full.score(X_train, y_train), 3), "test", round(full.score(X_test, y_test), 3))

tree3 = DecisionTreeClassifier(max_depth=3, random_state=42)
tree3.fit(X_train, y_train)
print("depth 3 train", round(tree3.score(X_train, y_train), 3), "test", round(tree3.score(X_test, y_test), 3))

full depth 15
train 1.0 test 0.672
depth 3 train 0.822 test 0.778


### Task 3.3 – Overfitting

The full-depth tree scores 1.00 on training but only 0.672 on test, below the 0.7525 baseline, so it memorised noise in the training rows; limiting depth to 3 shrinks the gap (0.822 vs 0.778) and generalises much better.

In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)
rf_prob = rf.predict_proba(X_test)[:, 1]

print("train", round(rf.score(X_train, y_train), 3))
print("accuracy", round(accuracy_score(y_test, rf_pred), 3))
print("recall", round(recall_score(y_test, rf_pred), 2))
print("roc_auc", round(roc_auc_score(y_test, rf_prob), 3))

imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print(imp.head(3).round(2))

train 1.0
accuracy 0.758
recall 0.35
roc_auc 0.7
lead_days      0.22
age            0.20
distance_km    0.19
dtype: float64


In [6]:
tree3_pred = tree3.predict(X_test)
tree3_prob = tree3.predict_proba(X_test)[:, 1]

table = pd.DataFrame({
    "model": ["Logistic regression", "Decision tree (depth 3)", "Random forest"],
    "accuracy": [accuracy_score(y_test, pred), accuracy_score(y_test, tree3_pred), accuracy_score(y_test, rf_pred)],
    "recall": [recall_score(y_test, pred), recall_score(y_test, tree3_pred), recall_score(y_test, rf_pred)],
    "roc_auc": [roc_auc_score(y_test, prob), roc_auc_score(y_test, tree3_prob), roc_auc_score(y_test, rf_prob)],
})
print(table.round(3))

                     model  accuracy  recall  roc_auc
0      Logistic regression     0.788   0.347    0.789
1  Decision tree (depth 3)     0.778   0.224    0.700
2            Random forest     0.758   0.347    0.700


### Task 3.4 – Comparison

On the test set, logistic regression looks best: it has the highest accuracy (0.788) and ROC-AUC (0.789) and ties the forest on recall. The random forest scores 1.00 on training but drops to 0.758 on test, so it is still overfitting; the depth-3 tree catches the fewest no-shows (recall 0.224). This table is for reporting only; the formal choice is made in Task 3.5 using training data alone.

In [7]:
from sklearn.model_selection import GridSearchCV

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
    cv=5,
    scoring="roc_auc",
)
grid.fit(X_train, y_train)

print(grid.best_params_)
print(round(grid.best_score_, 3))

{'max_depth': 3, 'n_estimators': 200}
0.767


In [8]:
from sklearn.model_selection import cross_val_score, cross_val_predict
from sklearn.pipeline import Pipeline

lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])

cv_lr = cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_tree = cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()
cv_rf = cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42), X_train, y_train, cv=5, scoring="roc_auc").mean()

print("logistic regression", round(cv_lr, 3))
print("depth-3 tree", round(cv_tree, 3))
print("random forest", round(cv_rf, 3))
print("tuned forest", round(grid.best_score_, 3))

logistic regression 0.774
depth-3 tree 0.678
random forest 0.745
tuned forest 0.767


In [9]:
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]

for t in [0.5, 0.35]:
    flag = (oof >= t).astype(int)
    print(t,
          "precision", round(precision_score(y_train, flag), 2),
          "recall", round(recall_score(y_train, flag), 2),
          "f1", round(f1_score(y_train, flag), 2),
          "flagged", flag.sum())

0.5 precision 0.58 recall 0.25 f1 0.35 flagged 73
0.35 precision 0.51 recall 0.47 f1 0.49 flagged 154


In [10]:
threshold = 0.35

lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]
flag = (probs >= threshold).astype(int)

print("precision", round(precision_score(y_test, flag), 2))
print("recall", round(recall_score(y_test, flag), 2))
print("flagged", flag.sum())
print("roc_auc", round(roc_auc_score(y_test, probs), 3))

precision 0.57
recall 0.57
flagged 49
roc_auc 0.789


In [11]:
scores = pd.DataFrame({
    "appointment_id": df.loc[X_test.index, "appointment_id"],
    "no_show": y_test,
    "risk_prob": probs.round(4),
    "flag": flag,
})
scores.to_csv("../outputs/test_risk_scores.csv", index=False)

print(scores.shape)
print(scores["flag"].sum())
print(scores.head())

(198, 4)
49
    appointment_id  no_show  risk_prob  flag
213          A0217        0     0.0216     0
331          A0336        0     0.1509     0
501          A0508        0     0.5187     1
309          A0314        0     0.1913     0
88           A0089        0     0.1677     0


### Task 3.5 – Model and threshold choice

Logistic regression had the highest cross-validated ROC-AUC on the training set (0.774, vs tuned forest 0.767, forest 0.745, depth-3 tree 0.678), so it is the final model. Using out-of-fold training probabilities, threshold 0.35 gave a higher F1 (0.49 vs 0.35) and much higher recall (0.47 vs 0.25) than 0.5, so 0.35 was chosen. Recall matters here because a missed warning costs an empty slot, while an extra call only costs the front desk a minute.

Tested once on the test set at 0.35: precision 0.57, recall 0.57, 49 flagged, ROC-AUC 0.789. Scores saved in outputs/test_risk_scores.csv.

In practice a clinic would set the threshold by how many follow-up calls the front desk can make in an evening, weighed against the cost of an empty slot.